In [1]:
import boto3
import boto3
import duckdb
import tempfile
from datetime import datetime

In [2]:
def extract_timestamp(folder_name):
    timestamp_str = folder_name.rstrip("/").replace("ingest_date_", "")
    return datetime.strptime(timestamp_str, "%Y%m%d_%H%M%S")

In [3]:
def load_valid_folders(start_date,end_date,bucket_name,prefix) :
    valid_folder = {}

    response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix = prefix ,
    Delimiter="/"
    )

    for folder in response.get("CommonPrefixes", []):
        folder_name = folder["Prefix"].split("/")[-2]

        folder = f"{folder_name}/"
        ts = extract_timestamp(folder)

        if ts >= start_date and ts <end_date:
            valid_folder[folder_name]=ts
            print(folder_name)

        else:
            pass

    return valid_folder
    

In [14]:
dict = {1:"a"}
if 2 not in dict:
    print("tset")

tset


In [18]:
def load_master_schema():
    with open("master_schema.json", "r") as f:
        schema_dict = json.load(f)
    return schema_dict

def update_scheme_master(column_names, schema_dict):

    columns_added = 0 
    for col in column_names :
        if col not in schema_dict :
            schema_dict[col] = "VARCHAR"
            print("New Column Found : ", col)
            columns_added+=1

    if columns_added ==0 :
        pass
    else:
        try:
            json.dumps(schema_dict)
        
            with open("master_schema.json", "w") as f:
                json.dump(schema_dict, f, indent=4)

        except:
            print("Error")            
    
schema_dict= load_master_schema()
column_names = ["unique_key","test"]
update_scheme_master(column_names,schema_dict)
schema_dict= load_master_schema()
print(schema_dict)

New Column Found :  test
{'unique_key': 'VARCHAR', 'created_date': 'TIMESTAMP', 'closed_date': 'TIMESTAMP', 'agency': 'VARCHAR', 'agency_name': 'VARCHAR', 'complaint_type': 'VARCHAR', 'descriptor': 'VARCHAR', 'descriptor_2': 'VARCHAR', 'incident_zip': 'VARCHAR', 'incident_address': 'VARCHAR', 'street_name': 'VARCHAR', 'cross_street_1': 'VARCHAR', 'cross_street_2': 'VARCHAR', 'address_type': 'VARCHAR', 'city': 'VARCHAR', 'status': 'VARCHAR', 'resolution_description': 'VARCHAR', 'resolution_action_updated_date': 'TIMESTAMP', 'community_board': 'VARCHAR', 'council_district': 'INTEGER', 'police_precinct': 'INTEGER', 'bbl': 'VARCHAR', 'borough': 'VARCHAR', 'x_coordinate_state_plane': 'DOUBLE', 'y_coordinate_state_plane': 'DOUBLE', 'open_data_channel_type': 'VARCHAR', 'park_facility_name': 'VARCHAR', 'park_borough': 'VARCHAR', 'latitude': 'DOUBLE', 'longitude': 'DOUBLE', 'location': 'VARCHAR', 'facility_type': 'VARCHAR', 'intersection_street_1': 'VARCHAR', 'intersection_street_2': 'VARCHAR',

In [ ]:
select_expr = []

for col, dtype in master_schema.items():
    if col in current_columns:
        select_expr.append(
            f'CAST("{col}" AS {dtype}) AS "{col}"'
        )
    else:
        select_expr.append(
            f'CAST(NULL AS {dtype}) AS "{col}"'
        )

select_expr = ",\n".join(select_expr)

In [4]:
def load_data_to_duckdb(valid_folder,Prefix):
    #connect to s3 bucket
    s3 = boto3.client("s3")
    bucket = "nyi-data-analytics"
    
    #connect to DuckDb
    con = duckdb.connect("bronze_staging_db.duckdb")

    #drop the old table (to fetch & clean only the new_data) & Creates One if First Run
    con.execute("DROP TABLE IF EXISTS staging_bronze;")

    files_pushed = 0

    for folder_name, ts in valid_folder.items():

        prefix = f"{Prefix}{folder_name}/"

        # list parquet files
        resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

        #we will traverse each parquet file , type cast it & upload to duckdb for transformations
        for obj in resp.get("Contents", []):
            key = obj["Key"]
    
            # download file
            file_obj = s3.get_object(Bucket=bucket, Key=key)

            #write file data on temp file
            with tempfile.NamedTemporaryFile(suffix=".parquet") as f:
                f.write(file_obj["Body"].read())
                f.flush()
    
                print("Fetched Data from location : ",key)

                #read columns to type cast them
                schema = con.execute("""
                    DESCRIBE SELECT * FROM read_parquet(?)
                    limit 0
                    """, [f.name]).fetchdf()
                
                column_names = schema["column_name"].tolist()
    
                select_expr = ",\n".join(
                    [f'CAST("{col}" AS VARCHAR) AS "{col}"' for col in column_names]
                )         

                con.execute(f"""
                CREATE TABLE IF NOT EXISTS staging_bronze AS
                SELECT {select_expr},
                       ? AS ingest_timestamp
                FROM read_parquet(?,union_by_name=true)
                limit 0
                """,[ts,f.name])
                
                con.execute(f"""
                INSERT INTO staging_bronze
                SELECT
                    {select_expr},
                    ? AS ingest_timestamp
                FROM read_parquet(?, union_by_name=true)
                """, [ts, f.name])

                files_pushed += 1

    print(files_pushed)
    return
                   

In [5]:
s3 = boto3.client("s3")

bucket_name = "nyi-data-analytics"
Prefix="bronze/full_load/"
#prefix ko abhi bronze/full_load karna hai

#the dates here are ingestion timestamp
start_date = datetime.strptime("2026-07-01 10:29:00", "%Y-%m-%d %H:%M:%S")
end_date = datetime.strptime("2026-07-03 00:00:00", "%Y-%m-%d %H:%M:%S")

#load valid folders (basically we want to fetch only the data that is required for staging not old data
valid_folder=load_valid_folders(start_date,end_date,bucket_name,Prefix)

load_data_to_duckdb(valid_folder,Prefix)

print()

ingest_date_20260702_091030
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_001.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_002.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_003.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_004.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_005.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_006.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_007.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_008.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_009.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_010.parquet
Fetched Data from location :  bronze/full_load/inges

BinderException: Binder Error: table staging_bronze has 45 columns but 43 values were supplied

In [13]:

start_date = datetime.strptime("2026-07-01 10:29:00", "%Y-%m-%d %H:%M:%S")
end_date = datetime.strptime("2026-07-02 00:00:00", "%Y-%m-%d %H:%M:%S")

In [16]:
con = duckdb.connect("bronze_staging_db.duckdb")

{'ingest_date_20260701_102928': datetime.datetime(2026, 7, 1, 10, 29, 28), 'ingest_date_20260701_103044': datetime.datetime(2026, 7, 1, 10, 30, 44)}


FileNotFoundError: [Errno 2] No such file or directory: 'part_017.parquet'